# ISOM 839 · Session 4 — Network Optimization I: Transportation, Transshipment & Assignment

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Last week the model told you what everything is worth. Tonight the decisions move onto a **map**: plants, warehouses, hubs, and people become nodes; lanes and matches become arcs. One skeleton — flow in, flow out, cost per arc — covers a beverage distributor, a consulting firm's staffing, and India's 810-million-person food network.

In [ ]:
%pip install -q gurobipy
import gurobipy as gp
from gurobipy import GRB
print('gurobipy ready:', gp.gurobi.version())

---
## Part 1 — The transportation model: one variable per lane

**Northeast Beverages** runs three bottling plants and supplies four regional warehouses. Costs are dollars per case on each lane.

The professional pattern: keep the data in dictionaries, create variables with `addVars` over the cost keys, and write constraints as generator expressions. Twelve lanes or twelve thousand — the code below does not change.

In [ ]:
supply = {'Boston': 300, 'Hartford': 260, 'Albany': 240}
demand = {'Portland': 180, 'Worcester': 220, 'Providence': 190, 'Springfield': 160}

cost = {  # $ per case
    ('Boston', 'Portland'): 4, ('Boston', 'Worcester'): 2, ('Boston', 'Providence'): 3, ('Boston', 'Springfield'): 5,
    ('Hartford', 'Portland'): 7, ('Hartford', 'Worcester'): 3, ('Hartford', 'Providence'): 2, ('Hartford', 'Springfield'): 2,
    ('Albany', 'Portland'): 8, ('Albany', 'Worcester'): 5, ('Albany', 'Providence'): 6, ('Albany', 'Springfield'): 3,
}
print(f'Total supply {sum(supply.values())}  vs  total demand {sum(demand.values())}')

m = gp.Model('shipping_network')
ship = m.addVars(cost.keys(), name='ship')                       # one variable per lane
m.setObjective(ship.prod(cost), GRB.MINIMIZE)                    # sum of cost x flow
sup = m.addConstrs((ship.sum(p, '*') <= supply[p] for p in supply), 'supply')   # rows cap
dem = m.addConstrs((ship.sum('*', w) == demand[w] for w in demand), 'demand')   # columns fill
m.optimize()

print(f'\nCheapest plan: ${m.ObjVal:,.0f}')
for (p, w), v in ship.items():
    if v.X > 0.5:
        print(f'  {p:9s} -> {w:12s} {v.X:6.0f} cases')
for p in supply:
    print(f'  {p:9s} idle: {sup[p].Slack:.0f} cases')

**Read the plan like a manager.** Hartford → Springfield costs $2 and carries nothing; Albany → Springfield costs $3 and carries 160. Hartford's 260 cases are worth more in Providence. And Albany keeps 50 cases idle — supply (800) exceeds demand (750), so the model chose who sits out.

### 💬 Discussion
Before scrolling: which *warehouse* do you think is the most expensive to serve at the margin? Which *plant* would you expand first?

---
## Part 2 — Session 3 didn't end: prices on a map

Every constraint still carries a `.Pi`. On a network they read beautifully: the demand duals are the marginal cost to serve each city; the supply duals are what one more case of plant capacity saves.

In [ ]:
print('One more case of DEMAND would cost:')
for w in demand:
    print(f'  {w:12s} ${dem[w].Pi:.2f}')
print('\nOne more case of CAPACITY would save:')
for p in supply:
    print(f'  {p:9s} ${-sup[p].Pi:.2f}   (idle {sup[p].Slack:.0f})')

Portland costs $7 at the margin even though its cheapest lane is $4 — serving one more Portland case forces a cascade of re-routing elsewhere. Albany's extra capacity is worth $0: it already has cases nobody wants. **A shadow price is a system number, not a lane number.**

### Same code, 4,000 lanes

Anna Chakra is this skeleton with thousands of nodes. Prove to yourself that the *code* is already there — only the data grows. Fifty plants, eighty warehouses, four thousand lanes, random costs:

In [ ]:
import random, time
random.seed(839)
big_supply = {f'P{i:02d}': random.randint(800, 1200) for i in range(50)}
big_demand = {f'W{j:02d}': random.randint(400, 800) for j in range(80)}
# scale demand so total demand is 90% of total supply (a realistic buffer)
scale = 0.9 * sum(big_supply.values()) / sum(big_demand.values())
big_demand = {w: int(d * scale) for w, d in big_demand.items()}
big_cost = {(p, w): random.randint(1, 20) for p in big_supply for w in big_demand}
print(f'{len(big_cost):,} lanes · supply {sum(big_supply.values()):,} · demand {sum(big_demand.values()):,}')

t0 = time.time()
big = gp.Model('national_network'); big.Params.OutputFlag = 0
f = big.addVars(big_cost.keys(), name='f')                                     # identical lines
big.setObjective(f.prod(big_cost), GRB.MINIMIZE)                               # to Part 1 —
big.addConstrs((f.sum(p, '*') <= big_supply[p] for p in big_supply), 'supply')  # only the
big.addConstrs((f.sum('*', w) == big_demand[w] for w in big_demand), 'demand')  # data changed
big.optimize()
used = sum(1 for v in f.values() if v.X > 0.5)
frac = sum(1 for v in f.values() if abs(v.X - round(v.X)) > 1e-6)
print(f'Solved in {time.time() - t0:.2f}s · cost ${big.ObjVal:,.0f} · {used} of {len(big_cost):,} lanes used · fractional shipments: {frac}')

Under a second, roughly one lane per warehouse actually used, and **zero fractional shipments** — the integrality guarantee holds at scale, which is the whole reason a national program can re-plan monthly. (The free Colab license caps model size; this one fits. Anna Chakra's does not, which is what the academic license is for.)

---
## Part 3 — Unbalanced networks and the honest solver

Supply must cover demand, or the model is infeasible. Watch it tell you the truth:

In [ ]:
dem['Portland'].RHS = 230        # total demand now 800 = total supply
m.optimize()
print(f'Portland 230: ${m.ObjVal:,.0f}  ({m.Status == GRB.OPTIMAL and "every plant runs full"})')

dem['Portland'].RHS = 240        # total demand 810 > 800
m.optimize()
print('Portland 240: status =', 'INFEASIBLE' if m.Status == GRB.INFEASIBLE else m.Status)

dem['Portland'].RHS = 180        # put it back
m.optimize()

**Fix for a real shortage:** add a dummy "Shortage" source with a penalty cost per unmet case. The solver then decides *which* city goes short — the one whose marginal cost is highest. Try it: `supply['Shortage'] = 10`, cost $20 on every lane from it, rebuild the model.

---
## Part 4 — Transshipment: the one new idea

Northeast is considering a cross-dock in **Framingham**. Product can flow plant → Framingham → warehouse. Inbound costs: Boston $1, Hartford $1, Albany $2. Outbound: Portland $4, Worcester $1, Providence $1, Springfield $2.

One new constraint type unlocks it: at the hub, **what comes in must go out.**

In [ ]:
hub_in  = {'Boston': 1, 'Hartford': 1, 'Albany': 2}
hub_out = {'Portland': 4, 'Worcester': 1, 'Providence': 1, 'Springfield': 2}
handling_fee = 0     # $ per case through the hub — change me later

arc_cost = dict(cost)
arc_cost.update({(p, 'Framingham'): c for p, c in hub_in.items()})
arc_cost.update({('Framingham', w): c + handling_fee for w, c in hub_out.items()})

t = gp.Model('transshipment')
flow = t.addVars(arc_cost.keys(), name='flow')
t.setObjective(flow.prod(arc_cost), GRB.MINIMIZE)
t.addConstrs((flow.sum(p, '*') <= supply[p] for p in supply), 'supply')
t.addConstrs((flow.sum('*', w) == demand[w] for w in demand), 'demand')
balance = t.addConstr(flow.sum('*', 'Framingham') == flow.sum('Framingham', '*'), 'hub_balance')   # in = out
t.optimize()

print(f'\nWith the cross-dock: ${t.ObjVal:,.0f}   (direct-only plan was $2,180)')
for (a, b), v in flow.items():
    if v.X > 0.5:
        print(f'  {a:10s} -> {b:12s} {v.X:6.0f}')

The hub saves **$130 a day** — 100 cases from Hartford and Albany consolidate through Framingham into Worcester. Now set `handling_fee = 1` and re-run: the saving shrinks to $30. The model doesn't love hubs. It finds exactly when they pay.

**Try:** at what handling fee does the hub stop being used at all?

---
## Part 5 — Integer answers for free

We never declared integer variables, yet every shipment came out a whole number. That's total unimodularity: network constraint matrices have one +1 and one −1 per column, so every corner of the feasible region is integer **as long as the supplies and demands are.** Watch the guarantee depend on the data:

In [ ]:
dem['Providence'].RHS = 150.5
m.optimize()
print(f'Providence 150.5 -> ${m.ObjVal:,.1f}')
for (p, w), v in ship.items():
    if v.X > 1e-6:
        print(f'  {p:9s} -> {w:12s} {v.X:7.1f}')
dem['Providence'].RHS = 190
m.optimize()

Two lanes now carry fractions (150.5 and 9.5). **The fraction came from the data, never from the model.** Whole supplies and demands in, whole shipments out — no integer programming, no branch-and-bound, which is why a nation-scale network solves in seconds.

---
## Part 6 — Assignment: transportation with all ones

Four TAs, four sections, a fit score for each pair. Each TA gets exactly one section; each section gets exactly one TA. Supplies of 1, demands of 1 — so integrality-for-free makes every variable a clean 0 or 1.

In [ ]:
tas      = ['Priya', 'Marcus', 'Lin', 'Sofia']
sections = ['Mon AM', 'Mon PM', 'Wed AM', 'Wed PM']
fit = {('Priya','Mon AM'): 9, ('Priya','Mon PM'): 8, ('Priya','Wed AM'): 5, ('Priya','Wed PM'): 6,
       ('Marcus','Mon AM'): 9, ('Marcus','Mon PM'): 5, ('Marcus','Wed AM'): 4, ('Marcus','Wed PM'): 3,
       ('Lin','Mon AM'): 5, ('Lin','Mon PM'): 7, ('Lin','Wed AM'): 9, ('Lin','Wed PM'): 8,
       ('Sofia','Mon AM'): 8, ('Sofia','Mon PM'): 4, ('Sofia','Wed AM'): 6, ('Sofia','Wed PM'): 9}

a = gp.Model('ta_assignment')
x = a.addVars(fit.keys(), name='x')                                   # continuous, on purpose
a.setObjective(x.prod(fit), GRB.MAXIMIZE)
a.addConstrs((x.sum(i, '*') == 1 for i in tas), 'one_section_each')
a.addConstrs((x.sum('*', j) == 1 for j in sections), 'one_ta_each')
a.optimize()

print(f'\nBest total fit: {a.ObjVal:.0f}')
for (i, j), v in x.items():
    if v.X > 0.5:
        print(f'  {i:7s} -> {j}   (fit {fit[i, j]}, x = {v.X:.0f})')

# The by-hand plan: everyone takes their favorite open section, in order
taken, greedy = set(), 0
for i in tas:
    j = max((j for j in sections if j not in taken), key=lambda j: fit[i, j])
    taken.add(j); greedy += fit[i, j]
print(f'"Everyone picks their favorite in order" scores {greedy}')

The by-hand plan scores 32; the solver scores 35. Priya's favorite (Mon AM, 9) is also Marcus's *only* good option — so the optimum gives Priya her second choice. **Greedy is locally right and globally wrong.** That gap grows with the size of the team; in Homework #3 it's your job to measure it.

---
# Homework #3 — The Staffing Matchmaker 🧩

**Due before Session 5 (Wed Oct 7), on Canvas: this notebook completed + a short written paragraph.**

**Beacon Street Advisory**, a Boston consulting firm, must staff six client projects for the fall cycle with six consultants. The partners scored every consultant–project pair on fit (skills, industry experience, client history), 0–100:

| | Fintech | Hospital | Retail | Energy | Nonprofit | Biotech |
|---|---|---|---|---|---|---|
| **Amara** | 92 | 61 | 85 | 55 | 48 | 66 |
| **Ben** | 58 | 88 | 64 | 72 | 51 | 60 |
| **Chloe** | 75 | 57 | 69 | 63 | 45 | 90 |
| **Dev** | 70 | 66 | 58 | 91 | 62 | 54 |
| **Elena** | 52 | 74 | 83 | 49 | 88 | 58 |
| **Farid** | 90 | 79 | 57 | 80 | 57 | 56 |

1. **Formulate and solve** the assignment problem to maximize total fit, using **continuous** variables. Show every x is a clean 0 or 1 and state the total.
2. **Compare to the partners' plan.** By tradition, consultants pick their favorite open project in seniority order (Amara first, Farid last). Compute that plan's total fit. How much does the model gain, and *who* pays for the by-hand plan's mistake?
3. **One twist, your choice** — (a) Chloe cannot work on Biotech (conflict of interest), or (b) Biotech is postponed and Hospital now needs two consultants. Re-solve and write a short paragraph on who moved, why, and what the twist cost in total fit.

Every claim in the paragraph cites a number from your solve.

In [ ]:
consultants = ['Amara', 'Ben', 'Chloe', 'Dev', 'Elena', 'Farid']
projects    = ['Fintech', 'Hospital', 'Retail', 'Energy', 'Nonprofit', 'Biotech']
scores = [[92, 61, 85, 55, 48, 66],
          [58, 88, 64, 72, 51, 60],
          [75, 57, 69, 63, 45, 90],
          [70, 66, 58, 91, 62, 54],
          [52, 74, 83, 49, 88, 58],
          [90, 79, 57, 80, 57, 56]]
fit = {(c, p): scores[i][j] for i, c in enumerate(consultants) for j, p in enumerate(projects)}

hw = gp.Model('staffing')

# TODO 1: one continuous variable per (consultant, project) pair

# TODO 2: objective — maximize total fit

# TODO 3: each consultant exactly one project; each project exactly one consultant

hw.optimize()

# TODO 4: print the matching and the total; confirm every x is 0 or 1

# TODO 5: compute the partners' seniority-order plan and compare

# TODO 6: apply ONE twist, re-solve, and print who moved

---
### Submission checklist
- [ ] All TODO cells complete and running; every x printed as 0 or 1
- [ ] The partners' plan computed and compared (a number, not a feeling)
- [ ] One twist applied, re-solved, and explained in a short paragraph that cites the totals
- [ ] The Framingham handling-fee question in Part 4 answered (bonus)

**Next week — Session 5:** shortest paths, max flow, and supply chain design — the model decides which warehouses should exist at all. 🏭